# Practical 3: Clean the Data and Preprocess
**Course:** BE05000231 - PDS (Semester V, GTU GSET)
**Course Outcome:** CO2

### Objective:
Clean and preprocess structured log data: apply multi-pass URL decoding to expose evasion techniques, normalize paths without erasing attack traces, standardize missing fields, and generate data quality metrics.

## 1. Imports and Setup

In [ ]:
import sys
from pathlib import Path
sys.path.append('..')
import pandas as pd

from pipeline.parser import parse_file
from pipeline.cleaner import clean, deep_decode, normalize_path, data_quality_report

LOG_PATH = Path('../tests/fixtures/sample_access.log')


## 2. Deep URL Decoding Demonstration
Attackers often utilize double or triple URL-encoding (e.g. `%2527` -> `%27` -> `'`) to bypass single-pass web application firewalls.

In [ ]:
sample_evasion = 'cat%2527s%20OR%201%3D1--'
print('Raw Payload:          ', sample_evasion)
print('Single unquote:       ', deep_decode(sample_evasion, max_iter=1))
print('Deep decode (iter=3): ', deep_decode(sample_evasion, max_iter=3))


## 3. Path Normalization vs Attack Preservation
Path normalization collapses consecutive slashes and standardizes root aliases (`/index.html` -> `/`). However, raw and decoded URLs MUST be retained so traversal attacks (`../`) remain detectable.

In [ ]:
print('Normalized //api///v1/ :', normalize_path('//api///v1/'))
print('Normalized /INDEX.HTML  :', normalize_path('/INDEX.HTML'))


## 4. Run Cleaning Pipeline

In [ ]:
raw_df, _ = parse_file(LOG_PATH)
cleaned_df = clean(raw_df)

print(f'Cleaned DataFrame Rows: {len(cleaned_df)}')
cleaned_df[['ip', 'method', 'path', 'query', 'decoded_url', 'ua']].head(10)


## 5. Comprehensive Data Quality Audit

In [ ]:
report = data_quality_report(cleaned_df)
for k, v in report.items():
    print(f'{k}: {v}')


### Conclusion:
Practical 3 successfully resolved missing values, established timezone-aware chronological ordering, exposed obfuscated payloads via deep decoding, and generated comprehensive data quality reports.